In [ ]:
from IPython.display import HTML
HTML(open('../style.css', 'r').read())

# Handwritten Digit Recognition

This notebook has been adapted from the article
[Handwritten Digit Recognition Using PyTorch — Intro To Neural Networks](https://medium.com/@amitrajit_bose/handwritten-digit-mnist-pytorch-977b5338e627)
by [Amitrajit Bose](https://medium.com/@amitrajit_bose).

We begin by importing both `torch` and `torchvision`.

In [ ]:
%pip install torch torchvision

In [ ]:
import torch

### Download The Dataset & Define The Transforms

In [ ]:
from torchvision import datasets, transforms

Next, we define a *transformer* to normalize the data.
* First, our transformer transforms the data into a tensor.
* Then this tensor is normalized so that both the mean and the variance of the data is `0.5`.

In [ ]:
transformer = transforms.Compose([transforms.ToTensor(),
                                  transforms.Normalize((0.5,), (0.5,)),
                                ])

Download and load the training data.

In [ ]:
trainset = datasets.MNIST('Data/', download=True, train=True,  transform=transformer)
valset   = datasets.MNIST('Data/', download=True, train=False, transform=transformer)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
valloader   = torch.utils.data.DataLoader(valset,   batch_size=64, shuffle=True)

### Exploring The Data

We get the first batch of images together with its labels.

In [ ]:
dataiter = iter(trainloader)

In [ ]:
images, labels = dataiter.__next__()
print(type(images))
print(images.shape)
print(labels.shape)

We use the library `matplotlib` to plot the first image from this batch.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(images[0].numpy().squeeze(), cmap='gray_r');

In [ ]:
labels[0].item()

Next, we plot 60 images.

In [ ]:
figure = plt.figure()
num_of_images = 60
for index in range(1, num_of_images + 1):
    plt.subplot(6, 10, index)
    plt.axis('off')
    plt.imshow(images[index].numpy().squeeze(), cmap='gray_r')

### Defining The Neural Network

We will use a neural network with two hidden layers.  The first hidden layer has 128 neurons, the second layer has 64 neurons.

![https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/image/mlp_mnist.png](https://raw.githubusercontent.com/dmlc/web-data/master/mxnet/image/mlp_mnist.png)

In [ ]:
from torch import nn

In [ ]:
input_size   = 784
hidden_sizes = [128, 64]
output_size  = 10

We create a fully connected feed-forward neural network and use the ReLU function as activation function for the hidden layers.  The outputlayer uses the
`LogSoftmax` function.

In [ ]:
model = nn.Sequential(nn.Linear(input_size, hidden_sizes[0]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[0], hidden_sizes[1]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[1], output_size),
                      nn.LogSoftmax(dim=1))
print(model)

In [ ]:
criterion = nn.NLLLoss()

In [ ]:
help(nn.NLLLoss)

In [ ]:
from torch import optim

### Core Training Of Neural Network

In [ ]:
import time

In [ ]:
%%time
optimizer = optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
start     = time.time()
epochs    = 30
for e in range(epochs):
    running_loss = 0
    for images, labels in trainloader:
        # Flatten MNIST images into a 784 long vector
        images = images.view(images.shape[0], -1)
        # Training pass
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        #T his is where the model learns by backpropagating
        loss.backward()
        # And optimizes its weights here
        optimizer.step()
        running_loss += loss.item()
    else:
        print(f"Epoch {e:{2}d} - Training loss: {running_loss/len(trainloader)}")
stop = time.time()
print(f"\nTraining Time (in seconds) = {(stop-start)}")

In [ ]:
import numpy as np

Function for viewing an image and it's predicted classes.

In [ ]:
def view_classify(img, ps):

    ps = ps.data.numpy().squeeze()

    fig, (ax1, ax2) = plt.subplots(figsize=(6,9), ncols=2)
    ax1.imshow(img.resize_(1, 28, 28).numpy().squeeze())
    ax1.axis('off')
    ax2.barh(np.arange(10), ps)
    ax2.set_aspect(0.1)
    ax2.set_yticks(np.arange(10))
    ax2.set_yticklabels(np.arange(10))
    ax2.set_title('Class Probability')
    ax2.set_xlim(0, 1.1)
    plt.tight_layout()

In [ ]:
images, labels = next(iter(valloader))

img = images[1].view(1, 784)
# Turn off gradients to speed up this part
with torch.no_grad():
    logps = model(img)

# Output of the network are log-probabilities, need to take exponential for probabilities
ps = torch.exp(logps)
probab = list(ps.numpy()[0])
print("Predicted Digit =", probab.index(max(probab)))
view_classify(img.view(1, 28, 28), ps)

### Model Evaluation

In [ ]:
correct_count, all_count = 0, 0
for images,labels in valloader:
    for i in range(len(labels)):
        img = images[i].view(1, 784)
        # Turn off gradients to speed up this part
        with torch.no_grad():
            logps = model(img)

        # Output of the network are log-probabilities, need to take exponential for probabilities
        ps = torch.exp(logps)
        probab = list(ps.numpy()[0])
        pred_label = probab.index(max(probab))
        true_label = labels.numpy()[i]
        if true_label == pred_label:
            correct_count += 1
        all_count += 1

print("Number Of Images Tested =", all_count)
print("\nModel Accuracy =", (correct_count/all_count))